# Lab 2 | ARIMA + AIC/BIC + chẩn đoán phần dư

**Mục bình chọn:** ARIMA — Dataset 1 và ARIMA baseline Dataset 2

**Hạn bàn giao:** 25/10/2026, 20:00 — bàn giao mô hình/code fit cho phần forecast và so sánh.

> Bạn xây dựng và kiểm tra mô hình ARIMA: thử cấu hình, so AIC/BIC, kiểm tra hội tụ và phần dư. Ngoài mô hình chính của Dataset 1, bạn phải có ARIMA trên Dataset 2 để phần nâng cao so sánh với SARIMA.

**Đối chiếu đề:** Bài cơ bản: Bài 5 mục chia dữ liệu, thiết lập ARIMA và chẩn đoán phần dư; câu hỏi 1–5. Nâng cao: ARIMA baseline trên chính Dataset 2.

Đây là mô tả nhận việc, không phải lời giải. Các cell code vẫn giữ nguyên `# TODO`.

## Đầu vào, phụ thuộc và việc làm song song

**Cần nhận:** d/D và phép xử lý từ ADF; các cấu hình ARIMA ứng viên từ ACF/PACF.
**Có thể chuẩn bị trước:** khung fit/so sánh mô hình, bảng AIC/BIC và khung đồ thị phần dư.
Khi có cấu hình, ARIMA và SARIMA làm song song; bạn không phải chờ SARIMA fit xong.

## Cụ thể cần làm gì?

1. Xác nhận train/test 80/20 theo thời gian: Dataset 1 là 163/41, Dataset 2 là 115/29, không shuffle.
2. Fit 1–2 cấu hình ARIMA trên train1 gốc, ghi rõ (p,d,q), AIC, BIC và trạng thái hội tụ.
   Không đưa chuỗi đã sai phân vào rồi lại dùng d>0 để sai phân thêm lần nữa.
3. So sánh ứng viên bằng thông tin trong train và chẩn đoán; không dùng lỗi test để chọn cấu hình.
4. Vẽ phần dư theo thời gian và ACF phần dư; kiểm tra mean gần 0 và dấu hiệu tự tương quan.
   Nếu cần bổ sung kiểm định, ghi cách làm và diễn giải, không chỉ đưa bảng số.
5. Fit ARIMA baseline trên train2 để phần nâng cao có mô hình đối chiếu với SARIMA.
6. Trả lời câu hỏi 1–5 Bài 5: vì sao không shuffle, vai trò test, ý nghĩa p/d/q,
   hội tụ, white noise và hướng xử lý khi phần dư còn tự tương quan.

## Cần bàn giao những gì?

**Gửi cho phần forecast/đánh giá:**
- Code fit có thể chạy lại từ đầu, cấu hình cuối, thông tin dữ liệu/index và cách tạo forecast.
- Bảng AIC/BIC/hội tụ, đồ thị phần dư và kết luận chẩn đoán cho Dataset 1.
- ARIMA baseline Dataset 2 cùng cấu hình và code tái tạo; không bỏ sót phần này.
- Mốc test/horizon phải giữ nguyên: 41 tháng Dataset 1, 29 tháng Dataset 2.
- Không chỉ gửi tên biến fitted model trong kernel hoặc một file pickle phụ thuộc môi trường.
  Phần forecast chịu trách nhiệm tổng hợp đồ thị/metrics cuối; bạn không cần làm trùng toàn bộ bảng đánh giá.

## Quy ước chung

- Chỉ bổ sung code vào các cell `# TODO`; phần nhận xét viết ở cell Markdown ngay bên dưới.
- Đọc CSV chung trong `data/`; không sửa nguồn, không tạo một bản dataset khác.
- Dataset 1 (`a10.csv`) dùng `date`/`value`; Dataset 2 (`airline-passengers.csv`) dùng `Month`/`Passengers`.
- Dùng cùng cách chia **80% train / 20% test theo thứ tự thời gian, không shuffle**:
  Dataset 1 có **163 train / 41 test**; Dataset 2 có **115 train / 29 test**.
- Có thể dùng helper `src/lab2_data.py`, nhưng phần import/load vẫn để bạn tự viết.
  Chỉ lựa chọn phép biến đổi và tham số từ train; không dùng test để chọn mô hình.
- Notebook phải chạy được từ kernel mới; không dựa vào biến đang nằm ở notebook khác.
- Bàn giao qua file notebook/PR có code và nhận xét tái tạo được, không chỉ gửi ảnh chụp.

## Khi nào xem là hoàn thành?

- [x] Dataset 1 có bảng so sánh mô hình, hội tụ và chẩn đoán phần dư.
- [x] Có ARIMA baseline Dataset 2, không chỉ làm Dataset 1.
- [x] Code fit và cấu hình bàn giao đủ để tái tạo forecast.
- [x] Trả lời câu hỏi 1–5 Bài 5.

---
## Khu vực thực hiện
Tự bổ sung import, đọc dữ liệu và code vào các cell TODO; ghi kết quả/nhận xét ngay dưới từng phần.

### 1. Fit và so sánh ARIMA trên Dataset 1

Nhận các cấu hình từ ACF/PACF; fit trên train1 gốc.

In [ ]:
import pandas as pd
import warnings
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from src.lab2_data import load_datasets, chronological_split

# Đọc dữ liệu chuẩn theo hợp đồng dữ liệu của dự án
_, _, y1, y2 = load_datasets()
train1, test1 = chronological_split(y1)
train2, test2 = chronological_split(y2)

print(f"Dataset 1: train={len(train1)} ({train1.index[0]:%Y-%m} -> {train1.index[-1]:%Y-%m}), test={len(test1)} ({test1.index[0]:%Y-%m} -> {test1.index[-1]:%Y-%m})")
print(f"Dataset 2: train={len(train2)} ({train2.index[0]:%Y-%m} -> {train2.index[-1]:%Y-%m}), test={len(test2)} ({test2.index[0]:%Y-%m} -> {test2.index[-1]:%Y-%m})")

# Các cấu hình thử nghiệm nhận từ ACF/PACF (d=1 từ kết quả kiểm định ADF)
orders1 = [(1, 1, 1), (2, 1, 1)]

models1 = {}
results1 = []

for order in orders1:
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            fitted = ARIMA(train1, order=order).fit()

        converged = fitted.mle_retvals.get("converged", None)
        warning_text = "; ".join(str(w.message) for w in caught)

        models1[order] = fitted

        results1.append({
            "Model": f"ARIMA{order}",
            "p": order[0],
            "d": order[1],
            "q": order[2],
            "AIC": round(fitted.aic, 2),
            "BIC": round(fitted.bic, 2),
            "Converged": converged,
            "Warning": warning_text if warning_text else "None"
        })

    except Exception as e:
        results1.append({
            "Model": f"ARIMA{order}",
            "p": order[0],
            "d": order[1],
            "q": order[2],
            "AIC": None,
            "BIC": None,
            "Converged": False,
            "Warning": str(e)
        })

comparison1 = pd.DataFrame(results1)
print("\nBảng so sánh mô hình Dataset 1:")
print(comparison1.to_string(index=False))

**Nhận xét:**
- Hai mô hình được thử nghiệm trên tập train của Dataset 1 là `ARIMA(1,1,1)` và `ARIMA(2,1,1)` với $d=1$ (được bàn giao từ bài ADF để loại bỏ xu hướng).
- **Kết quả thực nghiệm:**
  - `ARIMA(1,1,1)`: AIC = 582.47, BIC = 591.73, hội tụ thành công (`Converged = True`).
  - `ARIMA(2,1,1)`: AIC = 584.46, BIC = 596.81, hội tụ thành công (`Converged = True`).
- Mô hình `ARIMA(1,1,1)` có cả chỉ số AIC và BIC thấp hơn so với `ARIMA(2,1,1)`, đồng thời có cấu trúc tham số tinh gọn hơn. Do đó, `ARIMA(1,1,1)` được ưu tiên lựa chọn làm mô hình tốt nhất để thực hiện chẩn đoán phần dư ở bước tiếp theo.

### 2. Chẩn đoán phần dư và giải thích

So sánh AIC/BIC, hội tụ và chẩn đoán phần dư.

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

# Chọn ứng viên có AIC thấp nhất từ các mô hình fit thành công
valid_models1 = {order: m for order, m in models1.items() if pd.notna(m.aic)}
best_order1 = min(valid_models1, key=lambda order: valid_models1[order].aic)
best_model1 = valid_models1[best_order1]
residuals1 = best_model1.resid.dropna()

print("Mô hình được chọn để chẩn đoán:", best_order1)
print(f"AIC: {best_model1.aic:.2f}, BIC: {best_model1.bic:.2f}")
print("Trạng thái hội tụ:", best_model1.mle_retvals.get("converged", None))
print(f"Trung bình phần dư: {residuals1.mean():.4f}")

# 1. Đồ thị phần dư theo thời gian
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(residuals1)
ax.axhline(0, color="grey", linestyle="--", lw=0.8)
ax.set_title(f"Residuals - ARIMA{best_order1} Dataset 1")
ax.set_xlabel("Thời gian")
ax.set_ylabel("Phần dư")
plt.tight_layout()
plt.show()

# 2. ACF của phần dư
fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(residuals1, lags=24, ax=ax, title=f"ACF của phần dư - ARIMA{best_order1} Dataset 1")
plt.tight_layout()
plt.show()

# 3. Kiểm định Ljung-Box tại lag 12 và 24
lb1 = acorr_ljungbox(residuals1, lags=[12, 24], return_df=True)
print("\nKết quả kiểm định Ljung-Box Dataset 1:")
print(lb1)

**Nhận xét chẩn đoán phần dư Dataset 1:**
- **Trung bình phần dư:** Giá trị mean $\approx 0.318$ (xấp xỉ 0 nhưng hơi lệch dương), cho thấy mô hình không bị lệch thiên vị quá lớn về mức trung bình.
- **Biểu đồ thời gian phần dư:** Biên độ dao động của phần dư có xu hướng tăng dần theo thời gian (dấu hiệu của heteroskedasticity / phương sai thay đổi theo thời gian).
- **ACF của phần dư:** Tồn tại các đỉnh nhọn (spikes) vượt rõ ra ngoài dải tin cậy 95% tại các độ trễ mùa vụ $12$ và $24$.
- **Kiểm định Ljung-Box:**
  - Lag 12: $\text{lb\_stat} = 129.71$, $p\text{-value} = 7.07 \times 10^{-22} < 0.05$.
  - Lag 24: $\text{lb\_stat} = 228.42$, $p\text{-value} = 2.99 \times 10^{-35} < 0.05$.
- **Kết luận:** Bác bỏ giả thuyết $H_0$ về tính độc lập của phần dư. Phần dư **không phải là White Noise**, vẫn còn cấu trúc tự tương quan mùa vụ chu kỳ 12 tháng chưa được giải thích. Điều này chứng minh mô hình ARIMA phi mùa vụ là chưa đủ cho chuỗi a10 và đòi hỏi mô hình SARIMA.

### 3. ARIMA baseline trên Dataset 2

Fit ARIMA baseline trên train2 cho so sánh với SARIMA.

In [ ]:
# Thử hai cấu hình ARIMA baseline cho Dataset 2 trên train2
orders2 = [(1, 1, 1), (2, 1, 1)]
models2 = {}
results2 = []

for order in orders2:
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            fitted = ARIMA(train2, order=order).fit()

        models2[order] = fitted
        results2.append({
            "Model": f"ARIMA{order}",
            "AIC": round(fitted.aic, 2),
            "BIC": round(fitted.bic, 2),
            "Converged": fitted.mle_retvals.get("converged", None),
            "Warning": "; ".join(str(w.message) for w in caught) if caught else "None"
        })
    except Exception as e:
        results2.append({
            "Model": f"ARIMA{order}",
            "AIC": None,
            "BIC": None,
            "Converged": False,
            "Warning": str(e)
        })

comparison2 = pd.DataFrame(results2)
print("Bảng so sánh ARIMA baseline Dataset 2:")
print(comparison2.to_string(index=False))

# Chẩn đoán nhanh phần dư ARIMA baseline Dataset 2
best_order2 = min(models2, key=lambda order: models2[order].aic)
best_model2 = models2[best_order2]
resid2 = best_model2.resid.dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(resid2)
axes[0].axhline(0, color="grey", ls="--")
axes[0].set_title(f"Residuals - ARIMA{best_order2} Dataset 2")
plot_acf(resid2, lags=24, ax=axes[1], title=f"ACF Residuals - ARIMA{best_order2} Dataset 2")
plt.tight_layout()
plt.show()

lb2 = acorr_ljungbox(resid2, lags=[12, 24], return_df=True)
print("\nKiểm định Ljung-Box ARIMA baseline Dataset 2:")
print(lb2)

**Nhận xét ARIMA baseline Dataset 2:**
- Trên tập train gồm 115 quan sát của Dataset 2 (AirPassengers), cấu hình `ARIMA(1,1,1)` đạt AIC = 1058.25 và BIC = 1066.45, hội tụ tốt và nhỉnh hơn `ARIMA(2,1,1)` (AIC = 1058.65, BIC = 1069.59).
- **Chẩn đoán phần dư:** Biểu đồ ACF phần dư và kiểm định Ljung-Box cho thấy $p\text{-value} \ll 0.05$ tại lag 12, 24. Phần dư còn dao động mùa vụ rất mạnh hình sóng sin rõ rệt.
- **Vai trò:** Cấu hình `ARIMA(1,1,1)` được chọn làm mô hình đối chứng (Baseline). Ở các bài sau, mô hình SARIMA sẽ được so sánh trực tiếp với baseline này để định lượng mức độ cải thiện dự báo khi tính đến yếu tố mùa vụ.

### 4. Bàn giao mô hình và cách dự báo

Bàn giao cấu hình, code fit và dự báo theo đúng index test.

In [ ]:
import numpy as np

# Dự báo trên tập test bằng mô hình tốt nhất đã fit
forecast1 = best_model1.forecast(steps=len(test1))
forecast1.index = test1.index

forecast2 = best_model2.forecast(steps=len(test2))
forecast2.index = test2.index

# Hàm tính các chỉ số đánh giá theo yêu cầu Đề bài (Bài 5)
def compute_metrics(actual, pred):
    actual, pred = np.array(actual), np.array(pred)
    mae = np.mean(np.abs(actual - pred))
    rmse = np.sqrt(np.mean((actual - pred) ** 2))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    return {"MAE": round(mae, 4), "RMSE": round(rmse, 4), "MAPE (%)": round(mape, 2)}

metrics_summary = pd.DataFrame([
    {"Dataset": "Dataset 1 (a10)", "Model": f"ARIMA{best_order1}", **compute_metrics(test1, forecast1)},
    {"Dataset": "Dataset 2 (AirPassengers)", "Model": f"ARIMA{best_order2}", **compute_metrics(test2, forecast2)}
])
print("Bảng đánh giá dự báo ARIMA trên tập test:")
print(metrics_summary.to_string(index=False))

# Trực quan hóa dự báo
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

axes[0].plot(train1, label="Train (Thực tế)", color="tab:blue")
axes[0].plot(test1, label="Test (Thực tế)", color="tab:green")
axes[0].plot(forecast1, label=f"ARIMA{best_order1} Dự báo", color="tab:red", linestyle="--")
axes[0].set_title(f"ARIMA{best_order1} Forecast vs Actual - Dataset 1 (a10)")
axes[0].set_ylabel("Doanh số")
axes[0].legend()

axes[1].plot(train2, label="Train (Thực tế)", color="tab:blue")
axes[1].plot(test2, label="Test (Thực tế)", color="tab:green")
axes[1].plot(forecast2, label=f"ARIMA{best_order2} Dự báo", color="tab:red", linestyle="--")
axes[1].set_title(f"ARIMA{best_order2} Forecast vs Actual - Dataset 2 (AirPassengers)")
axes[1].set_ylabel("Số hành khách")
axes[1].legend()

plt.tight_layout()
plt.show()

**Nhận xét đánh giá dự báo:**
- **Dataset 1:** Mô hình `ARIMA(1,1,1)` nắm bắt được xu hướng tăng trưởng tổng thể nhưng hoàn toàn bỏ lỡ các đỉnh đáy chu kỳ mùa vụ hàng năm. Các giá trị dự báo có dạng đường thẳng tăng dần phẳng, dẫn đến sai lệch lớn vào các tháng cao điểm cuối năm.
- **Dataset 2:** Đường dự báo của `ARIMA(1,1,1)` là một đường dốc nhẹ, không thể hiện được biến động mùa hè tăng cao và mùa đông giảm mạnh của ngành hàng không.

---

### Trả lời câu hỏi 1–5 Bài 5 (Theo đề bài Lab 2)

1. **Vì sao không được shuffle dữ liệu chuỗi thời gian? Test set dùng để làm gì?**
   - **Không được shuffle:** Chuỗi thời gian có đặc tính phụ thuộc tuần tự theo thời gian ($y_t$ phụ thuộc vào các giá trị quá khứ $y_{t-1}, y_{t-2}...$). Nếu shuffle ngẫu nhiên, ta sẽ phá vỡ cấu trúc tự tương quan, đánh mất quy luật thời gian và nghiêm trọng nhất là gây ra hiện tượng **rò rỉ dữ liệu (data leakage)** khi mô hình dùng dữ liệu tương lai để học và dự báo quá khứ.
   - **Vai trò Test set:** Dùng để đánh giá năng lực dự báo ngoài mẫu (**out-of-sample forecast performance**) một cách khách quan trên dữ liệu tương lai chưa từng được tiếp cận trong quá trình huấn luyện.

2. **Ý nghĩa của $p, d, q$ đã chọn?**
   - $d = 1$: Bậc sai phân thường nhằm triệt tiêu xu hướng tăng dần (trend), đưa chuỗi gốc về chuỗi dừng theo kết quả kiểm định ADF.
   - $p = 1$: Bậc tự hồi quy (Autoregressive - AR), phản ánh giá trị hiện tại phụ thuộc tuyến tính vào 1 bước trễ liền trước ($y_{t-1}$).
   - $q = 1$: Bậc trung bình trượt (Moving Average - MA), phản ánh giá trị hiện tại phụ thuộc vào sai số ngẫu nhiên của bước liền trước ($\varepsilon_{t-1}$).

3. **Mô hình có hội tụ không?**
   - **Có.** Cả hai mô hình `ARIMA(1,1,1)` và `ARIMA(2,1,1)` trên cả hai dataset đều báo trạng thái hội tụ thành công (`converged = True`) thông qua thuật toán tối ưu hóa ước lượng hợp lý cực đại (MLE), không gặp cảnh báo `ConvergenceWarning`.

4. **Residuals có phải white noise không?**
   - **Không phải White Noise.** Mặc dù trung bình phần dư xấp xỉ 0 (mean $\approx 0.318$ trên Dataset 1), nhưng kiểm định Ljung-Box tại lag 12 và 24 đều cho $p\text{-value} \ll 0.05$ (cỡ $10^{-22}$ và $10^{-35}$). Đồ thị ACF của phần dư vẫn xuất hiện các cột vượt ngưỡng tin cậy tại lag 12 và 24. Điều này chứng minh phần dư vẫn còn tự tương quan mạnh.

5. **Nếu residuals còn tự tương quan $\rightarrow$ điều chỉnh gì?**
   - Do chuỗi có tính mùa vụ lặp lại theo chu kỳ 12 tháng, giải pháp cốt lõi là chuyển từ mô hình ARIMA phi mùa vụ sang mô hình **SARIMA** $(p, d, q)(P, D, Q)_{12}$ với việc bổ sung sai phân mùa vụ ($D=1$) cùng các tham số mùa vụ ($P, Q$).
   - Nếu chuỗi không có mùa vụ, có thể tăng bậc $p, q$, xem xét bậc sai phân $d$, hoặc áp dụng phép biến đổi ổn định phương sai (như $\log$ hoặc Box-Cox).